# ohie_qje_joint_test

> 对应代码：`EconometricsCode/code_in_notes/Chap.13/ohie_qje_joint_test.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebooks/PyStata.md`。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.13`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。

In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.13")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

我们先清空数据与 frames，然后读入俄勒冈医疗保险实验（OHIE）的数据。该研究通过抽签（lottery）随机分配医疗保险申请资格，因此第一步需要验证随机化是否成功——即处理组与对照组在抽签前的基线特征上是否平衡。

In [ ]:
%%stata
clear frames
set more off
use ../datasets/OHIE_QJE.dta

我们把需要平衡的基线变量（出生年份、性别、是否讲英语、是否自我报名、首日报名、是否有电话、是否有邮政信箱、邮编是否为都会区等）存入局部宏 `lottery_list`，方便后续批量处理。

In [ ]:
%%stata
// 比较随机化之前的特征
local lottery_list "birthyear_list female_list english_list self_list first_day_list have_phone_list pobox_list zip_msa"

为了对多个基线变量做**联合检验**，我们把这些变量连同处理指示变量、权重、家户标识一起复制到名为 `joint_compare` 的新 frame 中，并切换过去。使用 frames 可以避免 `preserve`/`restore` 的繁琐，同时保留原始数据不受改动。

In [ ]:
%%stata
// 合在一起比较
frames put `lottery_list' treatment weight_12m household_id numhh_list, into(joint_compare)
frame change joint_compare

接下来把数据从宽表整理成长表：循环中把每个基线变量统一改名为 `outcome1`、`outcome2`……，再用 `reshape long` 堆叠成单一的结果变量 `outcome`，用 `varid` 标记该观测来自哪一个原始变量。长表结构让我们能用“一个回归 + 交互项”的方式同时估计处理组与对照组在所有基线变量上的差异。

In [ ]:
%%stata
local i=0
foreach v of varlist `lottery_list'{
	local i=`i'+1
	rename `v' outcome`i'
}
gen id=_n
reshape long outcome, i(id) j(varid)

我们动态地构造待检验的系数列表。由于共有 `i` 个基线变量，联合检验的原假设是：处理指示变量与每个 `varid` 的交互项系数全为零，即 `1.treatment#1.varid = … = 1.treatment#i.varid = 0`。用循环把这些系数名拼接进局部宏 `test_coefs`，再显示出来核对。

In [ ]:
%%stata
local test_coefs ""
forvalues j=1/`i'{
	local test_coefs "`test_coefs' 1.treatment#`j'.varid"
}
di "`test_coefs'"

最后做联合平衡性检验。回归中 `i.varid` 控制各基线变量自身的均值差异，`i.treatment#i.varid` 给出每个变量上处理组与对照组的差，`i.numhh_list#i.varid` 控制家庭规模（抽签按家庭进行）；标准误在家庭层面聚类。`test` 对所有交互项做 Wald 联合 F 检验，若不显著，说明随机化成功，处理组与对照组在基线上不可区分。检验完成后切回默认 frame，恢复原数据环境。

In [ ]:
%%stata
reg outcome i.varid i.treatment#i.varid i.numhh_list#i.varid , cl(household_id)
test `test_coefs'
frame change default